# QUANTIZATION ANALYSIS — MOSHIKO 2.0

## What This Notebook Does
Compress and analyze the Moshiko model through experiments:

| Experiment | Type | What | Why |
|------------|------|------|-----|
| 1 | Compare Existing | bf16 vs q8 (Kyutai's official) | Baseline quality/VRAM numbers |
| 2a | Manual INT8 PTQ | Simulate INT8 by quantizing weights in-place | Can you match Kyutai's quality? |
| 2b | INT4 PTQ | bitsandbytes NF4 weight quantization | Store model at 4-bit precision |
| 3a | Layer-wise Sensitivity | Quantize one layer at a time with manual INT8 | Which layers matter most? |
| 3b | Mixed Precision | Keep sensitive layers at bf16, quantize rest | Best quality/size tradeoff |
| 5a | LM Layer Sensitivity | Analyze 7B LM weight distributions | Which LM layers matter most? |
| 5b | LM INT4 | bitsandbytes Linear4bit on LM | Can we compress the big model? |

## Important: What Does NOT Work
- `torch.quantization.quantize_dynamic()` — incompatible with moshi's custom transformer architecture
- Dynamic quantization (Experiment 4 in original plan) — skipped
- Instead we use **manual weight quantization**: round weights to INT8, dequantize, run on GPU

## Checkpoint System
Every experiment saves results immediately. If Colab disconnects:
- Re-run from Cell 1 (setup)
- Completed experiments will auto-skip
- Resume from where you left off

## How to Add Your Own Audio Samples
Place WAV files (24kHz, mono) in Google Drive at:
`/content/drive/MyDrive/Moshiko2.0_Project/outputs/audio_input/`

Then reference them in Cell 5 by editing the `custom_audio_files` list.

---
## SETUP SECTION — Run Once Per Session
---

### Cell 1: Dependencies

In [ ]:
import subprocess, sys, importlib

def install(package, import_name=None):
    name = import_name or package.split("==")[0].replace("-", "_")
    try:
        importlib.import_module(name)
        print(f"  OK: {package}")
    except ImportError:
        print(f"  Installing: {package} ...", end="", flush=True)
        result = subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", package],
            capture_output=True, text=True
        )
        print(" Done" if result.returncode == 0 else f" FAILED")

print("Installing PyTorch 2.4.0 + CUDA 12.1...")
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "torch==2.4.0", "torchaudio==2.4.0",
    "--index-url", "https://download.pytorch.org/whl/cu121"
])
print("  Done")

print("\nInstalling moshi...")
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "git+https://github.com/kyutai-labs/moshi.git#egg=moshi&subdirectory=moshi"
])
print("  Done")

print("\nInstalling other dependencies...\n")
packages = [
    ("bitsandbytes>=0.41.0", "bitsandbytes"),
    ("soundfile", "soundfile"),
    ("numpy", "numpy"),
    ("scipy", "scipy"),
    ("matplotlib", "matplotlib"),
    ("safetensors", "safetensors"),
]
for pkg, imp in packages:
    install(pkg, imp)

print("\nVerifying critical imports...")
import torch
print(f"  torch: {torch.__version__}")
import moshi
print(f"  moshi: OK")
import bitsandbytes
print(f"  bitsandbytes: {bitsandbytes.__version__}")
print("\nRestart runtime now: Runtime -> Restart runtime")
print("Then re-run from Cell 2.")

### Cell 2: Create Drive Folder Structure

In [ ]:
import os
from datetime import datetime
from google.colab import drive

print("Mounting Google Drive...")
drive.mount("/content/drive", force_remount=False)

BASE_DIR = "/content/drive/MyDrive/Moshiko2.0_Project"

FOLDERS = {
    "base":           BASE_DIR,
    "models":         f"{BASE_DIR}/models",
    "models_bf16":    f"{BASE_DIR}/models/bf16",
    "models_q8":      f"{BASE_DIR}/models/q8",
    "checkpoints":    f"{BASE_DIR}/checkpoints",
    "outputs":        f"{BASE_DIR}/outputs",
    "audio_in":       f"{BASE_DIR}/outputs/audio_input",
    "audio_out":      f"{BASE_DIR}/outputs/audio_output",
    "benchmarks":     f"{BASE_DIR}/outputs/benchmarks",
    "comparisons":    f"{BASE_DIR}/outputs/comparisons",
    "logs":           f"{BASE_DIR}/logs",
    "quant_base":     f"{BASE_DIR}/quantization",
    "quant_models":   f"{BASE_DIR}/quantization/models",
    "quant_int8":     f"{BASE_DIR}/quantization/models/int8",
    "quant_int4":     f"{BASE_DIR}/quantization/models/int4",
    "quant_mixed":    f"{BASE_DIR}/quantization/models/mixed",
    "quant_results":  f"{BASE_DIR}/quantization/results",
    "quant_plots":    f"{BASE_DIR}/quantization/plots",
    "ft_base":        f"{BASE_DIR}/finetuning",
    "ft_dataset":     f"{BASE_DIR}/finetuning/dataset",
    "ft_checkpoints": f"{BASE_DIR}/finetuning/checkpoints",
    "ft_logs":        f"{BASE_DIR}/finetuning/logs",
    "ft_outputs":     f"{BASE_DIR}/finetuning/outputs",
}

for name, path in FOLDERS.items():
    os.makedirs(path, exist_ok=True)
    print(f"  OK: {name}")

import builtins
builtins.FOLDERS = FOLDERS
builtins.BASE_DIR = BASE_DIR

print(f"\nAll folders created/verified at: {BASE_DIR}")

### Cell 3: Initialize Checkpoint Manifest

In [ ]:
import json, os

MANIFEST_PATH = f"{FOLDERS['quant_results']}/checkpoint_manifest.json"

def load_manifest():
    if os.path.exists(MANIFEST_PATH):
        with open(MANIFEST_PATH, "r") as f:
            return json.load(f)
    return {
        "created_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "last_updated": None,
        "completed_experiments": [],
        "pending_experiments": [
            "exp1_mimi_comparison",
            "exp2_ptq_int8",
            "exp2_ptq_int8_comparison",
            "exp2_ptq_int4",
            "exp3_layerwise_sensitivity",
            "exp3_mixed_precision",
            "exp5_lm_layer_sensitivity",
            "exp5_lm_int4",
            "exp6_final_report",
        ],
        "results_files": {},
        "errors": {}
    }

def save_manifest(manifest):
    manifest["last_updated"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    os.makedirs(os.path.dirname(MANIFEST_PATH), exist_ok=True)
    with open(MANIFEST_PATH, "w") as f:
        json.dump(manifest, f, indent=2)

def mark_complete(manifest, exp_name, result_file=None):
    if exp_name not in manifest["completed_experiments"]:
        manifest["completed_experiments"].append(exp_name)
    if exp_name in manifest["pending_experiments"]:
        manifest["pending_experiments"].remove(exp_name)
    if result_file:
        manifest["results_files"][exp_name] = result_file
    save_manifest(manifest)

def is_complete(manifest, exp_name):
    return exp_name in manifest["completed_experiments"]

manifest = load_manifest()
save_manifest(manifest)

print(f"Manifest initialized at: {MANIFEST_PATH}")
print(f"Completed: {len(manifest['completed_experiments'])} experiments")
print(f"Pending: {len(manifest['pending_experiments'])} experiments")
if manifest["completed_experiments"]:
    print(f"Already done: {', '.join(manifest['completed_experiments'])}")

### Cell 4: GPU Check + Environment Info

In [ ]:
import torch, sys, json

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    raise SystemExit("NO GPU DETECTED. Enable GPU: Runtime -> Change runtime type -> T4")

DEVICE = "cuda"
gpu_name = torch.cuda.get_device_name(0)
vram_total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {gpu_name}")
print(f"VRAM: {vram_total_gb:.1f} GB")

torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

builtins.DEVICE = DEVICE
builtins.GPU_NAME = gpu_name
builtins.VRAM_TOTAL_GB = vram_total_gb

print(f"\nTF32 disabled (T4 compatibility)")
print(f"Environment ready.")

### Cell 5: Load Models + Generate Test Signals

In [ ]:
import torch, os, time, numpy as np, soundfile as sf
from moshi.models import loaders, LMGen

DEVICE = builtins.DEVICE
FOLDERS = builtins.FOLDERS
SAMPLE_RATE = 24000

Q8_DIR = FOLDERS["models_q8"]
BF16_DIR = FOLDERS["models_bf16"]

mimi_q8_path = os.path.join(Q8_DIR, "tokenizer-e351c8d8-checkpoint125.safetensors")
moshi_q8_path = os.path.join(Q8_DIR, "model.q8.safetensors")
mimi_bf16_path = os.path.join(BF16_DIR, "tokenizer-e351c8d8-checkpoint125.safetensors")

for label, path in [("Mimi q8", mimi_q8_path), ("Moshiko q8", moshi_q8_path), ("Mimi bf16", mimi_bf16_path)]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing: {label} at {path}\nRun moshiko2.0.ipynb Cell 5 first.")
    print(f"  Found: {label} ({os.path.getsize(path)/1e6:.0f} MB)")

print("\nLoading Mimi (q8)...")
mimi_q8 = loaders.get_mimi(mimi_q8_path, device=DEVICE)
mimi_q8.set_num_codebooks(8)
mimi_q8.eval()
print("  Loaded.")

print("Loading Mimi (bf16)...")
mimi_bf16 = loaders.get_mimi(mimi_bf16_path, device=DEVICE)
mimi_bf16.set_num_codebooks(8)
mimi_bf16.eval()
print("  Loaded.")

print("Loading Moshiko LM (q8)...")
lm_kwargs = {
    "delays": [0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1],
    "n_q": 16, "dep_q": 8, "card": 2048, "text_card": 32000,
    "existing_text_padding_id": 3, "dim": 4096, "num_heads": 32,
    "num_layers": 32, "hidden_scale": 4.125, "causal": True,
    "context": 3000, "max_period": 10000, "gating": "silu",
    "norm": "rms_norm_f32", "positional_embedding": "rope",
    "layer_scale": None,
    "depformer_dim": 1024, "depformer_dim_feedforward": 4224,
    "depformer_num_heads": 16, "depformer_num_layers": 6,
    "depformer_causal": True, "depformer_layer_scale": None,
    "depformer_multi_linear": True, "depformer_context": 8,
    "depformer_max_period": 10000, "depformer_gating": "silu",
    "depformer_pos_emb": "none", "depformer_weights_per_step": True,
    "quantize": True,
}
moshi_lm = loaders.get_moshi_lm(moshi_q8_path, device=DEVICE, lm_kwargs=lm_kwargs)
moshi_lm.eval()
print("  Loaded.")

builtins.mimi_q8 = mimi_q8
builtins.mimi_bf16 = mimi_bf16
builtins.moshi_lm = moshi_lm

torch.cuda.empty_cache()
vram_used = torch.cuda.memory_allocated() / 1e9
print(f"\n  VRAM used: {vram_used:.1f} GB")

print("\nGenerating test signals...")
duration = 3.0
t = torch.linspace(0, duration, int(SAMPLE_RATE * duration))

test_signals = {
    "pure_tone_440hz": (0.3 * torch.sin(2 * np.pi * 440 * t), "440Hz sine wave"),
    "mixed_tones": (0.2 * torch.sin(2 * np.pi * 200 * t) + 0.2 * torch.sin(2 * np.pi * 800 * t) + 0.2 * torch.sin(2 * np.pi * 1600 * t), "200+800+1600Hz"),
    "white_noise": (0.3 * torch.randn_like(t), "White noise"),
    "chirp_100_8000": (0.3 * torch.sin(2 * np.pi * (100 + 2633 * t) * t), "Chirp 100-8000Hz"),
}

custom_audio_files = []
audio_input_dir = FOLDERS["audio_in"]
for fname in custom_audio_files:
    fpath = os.path.join(audio_input_dir, fname)
    if os.path.exists(fpath):
        wav, sr = sf.read(fpath)
        if sr != SAMPLE_RATE:
            print(f"  WARNING: {fname} is {sr}Hz, expected {SAMPLE_RATE}Hz")
        wav_tensor = torch.tensor(wav, dtype=torch.float32)
        if wav_tensor.dim() == 2:
            wav_tensor = wav_tensor.mean(dim=1)
        if len(wav_tensor) > SAMPLE_RATE * 10:
            wav_tensor = wav_tensor[:SAMPLE_RATE * 10]
        test_signals[f"custom_{fname.replace('.wav', '')}"] = (wav_tensor, f"Custom: {fname}")
        print(f"  Loaded custom audio: {fname}")
    else:
        print(f"  MISSING custom audio: {fname}")

for name, (sig, desc) in test_signals.items():
    test_signals[name] = (sig.unsqueeze(0).unsqueeze(0), desc)

print(f"\nTest signals ready: {len(test_signals)}")
for name, (_, desc) in test_signals.items():
    print(f"  {name}: {desc}")

---
## EXPERIMENT 1: Compare Existing Checkpoints
Compare Kyutai's official bf16 vs q8 Mimi codec.

### Cell 6: Mimi Codec bf16 vs q8 Comparison

In [ ]:
import json, time, torch

manifest = load_manifest()
if is_complete(manifest, "exp1_mimi_comparison"):
    print("EXPERIMENT 1: Already completed. Skipping.")
    print(f"Results saved at: {manifest['results_files'].get('exp1_mimi_comparison', 'N/A')}")
else:
    print("EXPERIMENT 1: Mimi Codec bf16 vs q8 Comparison")
    print("="*60)

    mimi_bf16 = builtins.mimi_bf16
    mimi_q8 = builtins.mimi_q8
    results = {"experiment": "exp1_mimi_comparison", "date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"), "signals": {}}

    def compute_snr(original, reconstructed):
        original = original.cpu().flatten()
        reconstructed = reconstructed.cpu().flatten()
        min_len = min(len(original), len(reconstructed))
        original = original[:min_len]
        reconstructed = reconstructed[:min_len]
        noise = original - reconstructed
        signal_power = torch.mean(original ** 2)
        noise_power = torch.mean(noise ** 2)
        if noise_power < 1e-10:
            return float("inf")
        return float(10 * torch.log10(signal_power / noise_power))

    for name, (signal, desc) in test_signals.items():
        print(f"\nTesting: {name} ({desc})")
        signal = signal.to(DEVICE)

        torch.cuda.synchronize()
        t0 = time.time()
        with torch.no_grad():
            codes_bf16 = mimi_bf16.encode(signal)
            decoded_bf16 = mimi_bf16.decode(codes_bf16)
        torch.cuda.synchronize()
        time_bf16 = time.time() - t0
        snr_bf16 = compute_snr(signal, decoded_bf16)

        torch.cuda.synchronize()
        t0 = time.time()
        with torch.no_grad():
            codes_q8 = mimi_q8.encode(signal)
            decoded_q8 = mimi_q8.decode(codes_q8)
        torch.cuda.synchronize()
        time_q8 = time.time() - t0
        snr_q8 = compute_snr(signal, decoded_q8)

        vram_used = torch.cuda.memory_allocated() / 1e6

        results["signals"][name] = {
            "description": desc,
            "bf16": {"snr_db": round(snr_bf16, 2), "time_s": round(time_bf16, 4), "vram_mb": round(vram_used, 1)},
            "q8": {"snr_db": round(snr_q8, 2), "time_s": round(time_q8, 4), "vram_mb": round(vram_used, 1)},
            "snr_drop_db": round(snr_bf16 - snr_q8, 2) if snr_bf16 != float("inf") else "N/A",
        }
        print(f"  BF16: SNR={snr_bf16:.1f}dB, Time={time_bf16:.3f}s")
        print(f"  Q8:   SNR={snr_q8:.1f}dB, Time={time_q8:.3f}s")
        print(f"  Drop: {snr_bf16 - snr_q8:.1f}dB")

    result_file = f"{FOLDERS['quant_results']}/exp1_mimi_bf16_vs_q8.json"
    with open(result_file, "w") as f:
        json.dump(results, f, indent=2)
    mark_complete(manifest, "exp1_mimi_comparison", result_file)
    print(f"\nResults saved to: {result_file}")
    print("EXPERIMENT 1: COMPLETE")

### Cell 7: Plot Experiment 1 Results

In [ ]:
import json, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt, numpy as np

result_file = f"{FOLDERS['quant_results']}/exp1_mimi_bf16_vs_q8.json"
if not os.path.exists(result_file):
    print("Run Cell 6 first!")
else:
    with open(result_file, "r") as f:
        data = json.load(f)

    signals = list(data["signals"].keys())
    labels = [s.replace("_", " ").title() for s in signals]

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    snr_bf16 = [data["signals"][s]["bf16"]["snr_db"] for s in signals]
    snr_q8 = [data["signals"][s]["q8"]["snr_db"] for s in signals]
    x = np.arange(len(labels))
    w = 0.35
    axes[0].bar(x - w/2, snr_bf16, w, label="BF16", color="steelblue")
    axes[0].bar(x + w/2, snr_q8, w, label="Q8", color="coral")
    axes[0].set_ylabel("SNR (dB)")
    axes[0].set_title("Signal Quality (SNR)")
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(labels, rotation=30, ha="right")
    axes[0].legend()

    time_bf16 = [data["signals"][s]["bf16"]["time_s"] for s in signals]
    time_q8 = [data["signals"][s]["q8"]["time_s"] for s in signals]
    axes[1].bar(x - w/2, time_bf16, w, label="BF16", color="steelblue")
    axes[1].bar(x + w/2, time_q8, w, label="Q8", color="coral")
    axes[1].set_ylabel("Time (seconds)")
    axes[1].set_title("Encode+Decode Latency")
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(labels, rotation=30, ha="right")
    axes[1].legend()

    drops = []
    for s in signals:
        d = data["signals"][s]["snr_drop_db"]
        drops.append(d if d != "N/A" else 0)
    colors = ["red" if d > 2 else "orange" if d > 1 else "green" for d in drops]
    axes[2].bar(labels, drops, color=colors)
    axes[2].set_ylabel("SNR Drop (dB)")
    axes[2].set_title("Quality Loss from Q8")
    axes[2].tick_params(axis="x", rotation=30)

    plt.tight_layout()
    plot_path = f"{FOLDERS['quant_plots']}/exp1_quality_comparison.png"
    plt.savefig(plot_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Plot saved: {plot_path}")

    from IPython.display import Image, display
    display(Image(plot_path))

---
## EXPERIMENT 2: Manual Weight Quantization (Type 1)
We manually round weights to INT8 precision, then dequantize for inference.
This works on GPU (unlike torch.quantization.quantize_dynamic).

### Cell 8: Manual INT8 Weight Quantization on Mimi

In [ ]:
import torch, copy

manifest = load_manifest()
if is_complete(manifest, "exp2_ptq_int8"):
    print("EXPERIMENT 2a (INT8 PTQ): Already completed. Skipping.")
    print(f"Results saved at: {manifest['results_files'].get('exp2_ptq_int8', 'N/A')}")
else:
    print("EXPERIMENT 2a: Manual INT8 Weight Quantization on Mimi")
    print("="*60)
    print("Approach: Round each weight tensor to 8-bit, then dequantize.")
    print("This simulates INT8 precision while keeping the model runnable on GPU.\n")

    mimi_bf16 = builtins.mimi_bf16

    # Deep copy so we don't destroy the original
    mimi_int8 = copy.deepcopy(mimi_bf16).to(DEVICE)
    mimi_int8.eval()

    quantized_count = 0
    skipped_count = 0

    for name, module in mimi_int8.named_modules():
        if isinstance(module, (torch.nn.Linear, torch.nn.Conv1d, torch.nn.ConvTranspose1d)):
            with torch.no_grad():
                weight = module.weight.data
                scale = weight.abs().max() / 127.0
                if scale < 1e-10:
                    skipped_count += 1
                    continue
                # Round to INT8, then dequantize back to float
                q = torch.round(weight / scale).clamp(-128, 127)
                module.weight.data = q.to(weight.dtype) * scale

                if module.bias is not None:
                    bias = module.bias.data
                    bias_scale = bias.abs().max() / 127.0
                    if bias_scale >= 1e-10:
                        q_bias = torch.round(bias / bias_scale).clamp(-128, 127)
                        module.bias.data = q_bias.to(bias.dtype) * bias_scale

            quantized_count += 1

    print(f"Quantized: {quantized_count} layers")
    print(f"Skipped: {skipped_count} layers (near-zero weights)")

    model_path = f"{FOLDERS['quant_int8']}/mimi_manual_int8.pt"
    torch.save(mimi_int8.state_dict(), model_path)
    model_size_mb = os.path.getsize(model_path) / 1e6
    print(f"\nModel saved: {model_path} ({model_size_mb:.1f} MB)")

    builtins.mimi_int8 = mimi_int8
    result_file = f"{FOLDERS['quant_results']}/exp2_ptq_int8.json"
    meta = {"quantized_layers": quantized_count, "skipped": skipped_count, "model_size_mb": round(model_size_mb, 1)}
    with open(result_file, "w") as f:
        json.dump(meta, f, indent=2)
    mark_complete(manifest, "exp2_ptq_int8", result_file)
    print("EXPERIMENT 2a: COMPLETE — Run Cell 9 to compare")

### Cell 9: Compare Manual INT8 vs Kyutai Q8 vs BF16

In [ ]:
import json, time, torch

manifest = load_manifest()
if is_complete(manifest, "exp2_ptq_int8_comparison"):
    print("EXPERIMENT 2a Comparison: Already completed. Skipping.")
else:
    print("EXPERIMENT 2a: Comparing Manual INT8 vs Kyutai Q8 vs BF16")
    print("="*60)

    mimi_bf16 = builtins.mimi_bf16
    mimi_q8 = builtins.mimi_q8
    mimi_int8 = builtins.mimi_int8

    def compute_snr(original, reconstructed):
        original = original.cpu().flatten()
        reconstructed = reconstructed.cpu().flatten()
        min_len = min(len(original), len(reconstructed))
        original = original[:min_len]
        reconstructed = reconstructed[:min_len]
        noise = original - reconstructed
        signal_power = torch.mean(original ** 2)
        noise_power = torch.mean(noise ** 2)
        if noise_power < 1e-10:
            return float("inf")
        return float(10 * torch.log10(signal_power / noise_power))

    results = {"experiment": "exp2_ptq_int8_comparison", "date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"), "signals": {}}

    for name, (signal, desc) in test_signals.items():
        print(f"\nTesting: {name}")
        signal = signal.to(DEVICE)

        with torch.no_grad():
            t0 = time.time()
            codes_bf16 = mimi_bf16.encode(signal)
            decoded_bf16 = mimi_bf16.decode(codes_bf16)
            time_bf16 = time.time() - t0
            snr_bf16 = compute_snr(signal, decoded_bf16)

        with torch.no_grad():
            t0 = time.time()
            codes_q8 = mimi_q8.encode(signal)
            decoded_q8 = mimi_q8.decode(codes_q8)
            time_q8 = time.time() - t0
            snr_q8 = compute_snr(signal, decoded_q8)

        with torch.no_grad():
            t0 = time.time()
            codes_int8 = mimi_int8.encode(signal)
            decoded_int8 = mimi_int8.decode(codes_int8)
            time_int8 = time.time() - t0
            snr_int8 = compute_snr(signal, decoded_int8)

        results["signals"][name] = {
            "bf16": {"snr_db": round(snr_bf16, 2), "time_s": round(time_bf16, 4)},
            "kyutai_q8": {"snr_db": round(snr_q8, 2), "time_s": round(time_q8, 4)},
            "manual_int8": {"snr_db": round(snr_int8, 2), "time_s": round(time_int8, 4)},
            "kyutai_drop": round(snr_bf16 - snr_q8, 2) if snr_bf16 != float("inf") else "N/A",
            "manual_drop": round(snr_bf16 - snr_int8, 2) if snr_bf16 != float("inf") else "N/A",
        }
        print(f"  BF16: {snr_bf16:.1f}dB | Kyutai Q8: {snr_q8:.1f}dB | Manual INT8: {snr_int8:.1f}dB")

    result_file = f"{FOLDERS['quant_results']}/exp2_ptq_int8_comparison.json"
    with open(result_file, "w") as f:
        json.dump(results, f, indent=2)
    mark_complete(manifest, "exp2_ptq_int8_comparison", result_file)
    print(f"\nResults saved: {result_file}")
    print("EXPERIMENT 2a Comparison: COMPLETE")

### Cell 10: INT4 PTQ via bitsandbytes

In [ ]:
import torch, json, os
from safetensors.torch import save_file

manifest = load_manifest()
if is_complete(manifest, "exp2_ptq_int4"):
    print("EXPERIMENT 2b (INT4 PTQ): Already completed. Skipping.")
    print(f"Results saved at: {manifest['results_files'].get('exp2_ptq_int4', 'N/A')}")
else:
    print("EXPERIMENT 2b: INT4 PTQ via bitsandbytes NF4")
    print("="*60)
    print("Quantizing weights to 4-bit Normal Float using bitsandbytes.")
    print("This is storage-only — model cannot run inference with these weights.\n")

    mimi_bf16 = builtins.mimi_bf16
    import bitsandbytes.functional as F

    quantized_state = {}
    quantized_count = 0
    skipped_count = 0

    print("Quantizing weight matrices to INT4...")
    for name, param in mimi_bf16.named_parameters():
        if param.dim() >= 2 and param.numel() > 1024:
            try:
                qweight, absmax, quant_map = F.quantize_4bit(param.data, quant_type="nf4")
                quantized_state[name] = qweight.cpu()
                quantized_state[f"{name}.absmax"] = absmax.cpu()
                quantized_state[f"{name}.quant_map"] = quant_map.cpu()
                quantized_state[f"{name}.original_shape"] = torch.tensor(param.shape)
                quantized_state[f"{name}.quant_type"] = "nf4"
                quantized_count += 1
            except Exception as e:
                print(f"  SKIP {name}: {e}")
                quantized_state[name] = param.data.cpu()
                skipped_count += 1
        else:
            quantized_state[name] = param.data.cpu()

    model_path = f"{FOLDERS['quant_int4']}/mimi_int4.safetensors"
    save_file(quantized_state, model_path)
    model_size_mb = os.path.getsize(model_path) / 1e6
    print(f"\nQuantized: {quantized_count} layers, Skipped: {skipped_count}")
    print(f"INT4 model saved: {model_path} ({model_size_mb:.1f} MB)")

    meta = {
        "quant_type": "nf4",
        "quantized_layers": quantized_count,
        "skipped": skipped_count,
        "model_size_mb": round(model_size_mb, 1),
        "date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    }
    meta_path = f"{FOLDERS['quant_int4']}/mimi_int4_metadata.json"
    with open(meta_path, "w") as f:
        json.dump(meta, f, indent=2)

    result_file = f"{FOLDERS['quant_results']}/exp2_ptq_int4_metadata.json"
    with open(result_file, "w") as f:
        json.dump(meta, f, indent=2)
    mark_complete(manifest, "exp2_ptq_int4", result_file)
    print("EXPERIMENT 2b: COMPLETE")

---
## EXPERIMENT 3: Layer-wise Sensitivity Analysis
Quantize ONE layer at a time with manual INT8, measure quality drop.

### Cell 11: Identify and Categorize Mimi Layers

In [ ]:
import torch

mimi_bf16 = builtins.mimi_bf16

print("Mimi Model Structure:")
print("="*60)

layer_categories = {
    "encoder": [],
    "decoder": [],
    "bottleneck": [],
    "embedding": [],
    "other": [],
}

for name, module in mimi_bf16.named_modules():
    if isinstance(module, (torch.nn.Linear, torch.nn.Conv1d, torch.nn.ConvTranspose1d, torch.nn.LayerNorm)):
        n_params = sum(p.numel() for p in module.parameters())
        if "encoder" in name.lower() or "enc" in name.lower():
            layer_categories["encoder"].append((name, type(module).__name__, n_params))
        elif "decoder" in name.lower() or "dec" in name.lower():
            layer_categories["decoder"].append((name, type(module).__name__, n_params))
        elif "bottleneck" in name.lower() or "quantize" in name.lower() or "vq" in name.lower():
            layer_categories["bottleneck"].append((name, type(module).__name__, n_params))
        elif "embed" in name.lower():
            layer_categories["embedding"].append((name, type(module).__name__, n_params))
        else:
            layer_categories["other"].append((name, type(module).__name__, n_params))

total_params = 0
for category, layers in layer_categories.items():
    cat_params = sum(p[2] for p in layers)
    total_params += cat_params
    if layers:
        print(f"\n{category.upper()} ({len(layers)} layers, {cat_params:,} params):")
        for name, ltype, nparams in layers[:5]:
            print(f"  {name} ({ltype}, {nparams:,} params)")
        if len(layers) > 5:
            print(f"  ... and {len(layers)-5} more")

print(f"\nTotal quantizable params: {total_params:,}")

builtins.mimi_layer_categories = layer_categories
print("\nLayer categorization complete.")

### Cell 12: Quantize One Layer at a Time — Sensitivity Analysis

In [ ]:
import torch, json, copy, time

manifest = load_manifest()
if is_complete(manifest, "exp3_layerwise_sensitivity"):
    print("EXPERIMENT 3a (Layer-wise Sensitivity): Already completed. Skipping.")
    print(f"Results saved at: {manifest['results_files'].get('exp3_layerwise_sensitivity', 'N/A')}")
else:
    print("EXPERIMENT 3a: Layer-wise Sensitivity Analysis")
    print("="*60)
    print("Quantizing ONE layer at a time with manual INT8, measuring quality drop.")
    print("This will take a few minutes.\n")

    mimi_bf16 = builtins.mimi_bf16
    layer_categories = builtins.mimi_layer_categories

    def compute_snr(original, reconstructed):
        original = original.cpu().flatten()
        reconstructed = reconstructed.cpu().flatten()
        min_len = min(len(original), len(reconstructed))
        original = original[:min_len]
        reconstructed = reconstructed[:min_len]
        noise = original - reconstructed
        signal_power = torch.mean(original ** 2)
        noise_power = torch.mean(noise ** 2)
        if noise_power < 1e-10:
            return float("inf")
        return float(10 * torch.log10(signal_power / noise_power))

    def quantize_module_int8(module):
        with torch.no_grad():
            weight = module.weight.data
            scale = weight.abs().max() / 127.0
            if scale < 1e-10:
                return False
            q = torch.round(weight / scale).clamp(-128, 127)
            module.weight.data = q.to(weight.dtype) * scale
            if module.bias is not None:
                bias = module.bias.data
                bias_scale = bias.abs().max() / 127.0
                if bias_scale >= 1e-10:
                    q_bias = torch.round(bias / bias_scale).clamp(-128, 127)
                    module.bias.data = q_bias.to(bias.dtype) * bias_scale
        return True

    test_signal = test_signals["pure_tone_440hz"][0].to(DEVICE)
    with torch.no_grad():
        baseline_codes = mimi_bf16.encode(test_signal)
        baseline_decoded = mimi_bf16.decode(baseline_codes)
    baseline_snr = compute_snr(test_signal, baseline_decoded)
    print(f"Baseline BF16 SNR: {baseline_snr:.2f} dB\n")

    sensitivity_results = []
    all_layers = []
    for category, layers in layer_categories.items():
        all_layers.extend([(name, ltype, nparams, category) for name, ltype, nparams in layers])

    print(f"Testing {len(all_layers)} individual layers...\n")

    for i, (layer_name, layer_type, n_params, category) in enumerate(all_layers):
        try:
            model_copy = copy.deepcopy(mimi_bf16).to(DEVICE)

            module = None
            for name, mod in model_copy.named_modules():
                if name == layer_name:
                    module = mod
                    break

            if module is None or not isinstance(module, (torch.nn.Linear, torch.nn.Conv1d, torch.nn.ConvTranspose1d)):
                continue

            success = quantize_module_int8(module)
            if not success:
                continue

            with torch.no_grad():
                codes = model_copy.encode(test_signal)
                decoded = model_copy.decode(codes)
            snr = compute_snr(test_signal, decoded)
            snr_drop = baseline_snr - snr if snr != float("inf") else 0.0

            sensitivity_results.append({
                "layer_name": layer_name,
                "layer_type": layer_type,
                "category": category,
                "num_params": n_params,
                "snr_db": round(snr, 2) if snr != float("inf") else "inf",
                "snr_drop_db": round(snr_drop, 2),
            })

            if (i + 1) % 10 == 0 or i == len(all_layers) - 1:
                print(f"  Progress: {i+1}/{len(all_layers)} layers tested")

            del model_copy
            torch.cuda.empty_cache()

        except Exception as e:
            print(f"  ERROR on {layer_name}: {e}")
            sensitivity_results.append({
                "layer_name": layer_name,
                "layer_type": layer_type,
                "category": category,
                "num_params": n_params,
                "error": str(e),
            })

    sensitivity_results.sort(key=lambda x: x.get("snr_drop_db", 0), reverse=True)

    result_file = f"{FOLDERS['quant_results']}/exp3_layerwise_sensitivity.json"
    output = {
        "experiment": "exp3_layerwise_sensitivity",
        "date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "baseline_snr_db": round(baseline_snr, 2),
        "test_signal": "pure_tone_440hz",
        "total_layers_tested": len(sensitivity_results),
        "results": sensitivity_results,
    }
    with open(result_file, "w") as f:
        json.dump(output, f, indent=2)
    mark_complete(manifest, "exp3_layerwise_sensitivity", result_file)

    print(f"\nResults saved: {result_file}")
    print(f"\nTop 5 most sensitive layers:")
    for r in sensitivity_results[:5]:
        print(f"  {r['layer_name']}: {r['snr_drop_db']:.2f} dB drop ({r['category']}/{r['layer_type']})")
    print("\nEXPERIMENT 3a: COMPLETE")

### Cell 13: Plot Layer-wise Sensitivity

In [ ]:
import json, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt, numpy as np

result_file = f"{FOLDERS['quant_results']}/exp3_layerwise_sensitivity.json"
if not os.path.exists(result_file):
    print("Run Cell 12 first!")
else:
    with open(result_file, "r") as f:
        data = json.load(f)

    results = data["results"]
    valid = [r for r in results if "error" not in r]
    valid.sort(key=lambda x: x["snr_drop_db"], reverse=True)

    top_n = min(20, len(valid))
    top = valid[:top_n]

    fig, axes = plt.subplots(1, 2, figsize=(20, 6))

    labels = [r["layer_name"].split(".")[-1] if len(r["layer_name"].split(".")) > 2 else r["layer_name"] for r in top]
    drops = [r["snr_drop_db"] for r in top]
    colors = ["red" if d > 5 else "orange" if d > 2 else "green" for d in drops]

    axes[0].barh(range(len(labels)), drops, color=colors)
    axes[0].set_yticks(range(len(labels)))
    axes[0].set_yticklabels(labels, fontsize=8)
    axes[0].invert_yaxis()
    axes[0].set_xlabel("SNR Drop (dB)")
    axes[0].set_title(f"Top {top_n} Most Sensitive Layers (higher = more sensitive)")
    axes[0].axvline(x=2, color="orange", linestyle="--", alpha=0.5, label="Medium sensitivity")
    axes[0].axvline(x=5, color="red", linestyle="--", alpha=0.5, label="High sensitivity")
    axes[0].legend()

    categories = {}
    for r in valid:
        cat = r["category"]
        if cat not in categories:
            categories[cat] = []
        categories[cat].append(r["snr_drop_db"])

    cat_names = list(categories.keys())
    cat_means = [np.mean(categories[c]) for c in cat_names]
    cat_max = [max(categories[c]) for c in cat_names]

    x = np.arange(len(cat_names))
    w = 0.35
    axes[1].bar(x - w/2, cat_means, w, label="Avg SNR Drop", color="steelblue")
    axes[1].bar(x + w/2, cat_max, w, label="Max SNR Drop", color="coral")
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(cat_names, rotation=30, ha="right")
    axes[1].set_ylabel("SNR Drop (dB)")
    axes[1].set_title("Sensitivity by Layer Category")
    axes[1].legend()

    plt.tight_layout()
    plot_path = f"{FOLDERS['quant_plots']}/exp3_layerwise_sensitivity.png"
    plt.savefig(plot_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Plot saved: {plot_path}")

    from IPython.display import Image, display
    display(Image(plot_path))

### Cell 14: Mixed Precision Strategy

In [ ]:
import json, torch, copy, time

manifest = load_manifest()
if is_complete(manifest, "exp3_mixed_precision"):
    print("EXPERIMENT 3b (Mixed Precision): Already completed. Skipping.")
    print(f"Results saved at: {manifest['results_files'].get('exp3_mixed_precision', 'N/A')}")
else:
    print("EXPERIMENT 3b: Mixed Precision Strategy")
    print("="*60)
    print("Keep most sensitive layers at bf16, quantize rest to INT8.\n")

    result_file_sensitivity = f"{FOLDERS['quant_results']}/exp3_layerwise_sensitivity.json"
    if not os.path.exists(result_file_sensitivity):
        print("ERROR: Run Cell 12 first!")
    else:
        with open(result_file_sensitivity, "r") as f:
            sensitivity_data = json.load(f)

        valid_results = [r for r in sensitivity_data["results"] if "error" not in r]
        valid_results.sort(key=lambda x: x["snr_drop_db"], reverse=True)

        n_sensitive = max(1, len(valid_results) // 5)
        sensitive_layers = [r["layer_name"] for r in valid_results[:n_sensitive]]
        quantizable_layers = [r["layer_name"] for r in valid_results[n_sensitive:]]

        print(f"Total layers: {len(valid_results)}")
        print(f"Keeping at BF16 (top 20%): {n_sensitive} layers")
        print(f"Quantizing to INT8: {len(quantizable_layers)} layers")
        print(f"BF16 layers: {sensitive_layers[:3]}...")

        def quantize_module_int8(module):
            with torch.no_grad():
                weight = module.weight.data
                scale = weight.abs().max() / 127.0
                if scale < 1e-10:
                    return False
                q = torch.round(weight / scale).clamp(-128, 127)
                module.weight.data = q.to(weight.dtype) * scale
                if module.bias is not None:
                    bias = module.bias.data
                    bias_scale = bias.abs().max() / 127.0
                    if bias_scale >= 1e-10:
                        q_bias = torch.round(bias / bias_scale).clamp(-128, 127)
                        module.bias.data = q_bias.to(bias.dtype) * bias_scale
            return True

        mimi_bf16 = builtins.mimi_bf16
        mimi_mixed = copy.deepcopy(mimi_bf16).to(DEVICE)
        mimi_mixed.eval()

        quantized_count = 0
        for name, module in mimi_mixed.named_modules():
            if name in quantizable_layers and isinstance(module, (torch.nn.Linear, torch.nn.Conv1d, torch.nn.ConvTranspose1d)):
                if quantize_module_int8(module):
                    quantized_count += 1

        model_path = f"{FOLDERS['quant_mixed']}/mimi_mixed_precision.pt"
        torch.save(mimi_mixed.state_dict(), model_path)
        model_size_mb = os.path.getsize(model_path) / 1e6

        def compute_snr(original, reconstructed):
            original = original.cpu().flatten()
            reconstructed = reconstructed.cpu().flatten()
            min_len = min(len(original), len(reconstructed))
            original = original[:min_len]
            reconstructed = reconstructed[:min_len]
            noise = original - reconstructed
            signal_power = torch.mean(original ** 2)
            noise_power = torch.mean(noise ** 2)
            if noise_power < 1e-10:
                return float("inf")
            return float(10 * torch.log10(signal_power / noise_power))

        test_signal = test_signals["pure_tone_440hz"][0].to(DEVICE)
        with torch.no_grad():
            codes = mimi_mixed.encode(test_signal)
            decoded = mimi_mixed.decode(codes)
        snr_mixed = compute_snr(test_signal, decoded)

        with torch.no_grad():
            codes_bf16 = mimi_bf16.encode(test_signal)
            decoded_bf16 = mimi_bf16.decode(codes_bf16)
        snr_bf16 = compute_snr(test_signal, decoded_bf16)

        mimi_q8 = builtins.mimi_q8
        with torch.no_grad():
            codes_q8 = mimi_q8.encode(test_signal)
            decoded_q8 = mimi_q8.decode(codes_q8)
        snr_q8 = compute_snr(test_signal, decoded_q8)

        vram_mixed = torch.cuda.memory_allocated() / 1e6

        mixed_results = {
            "experiment": "exp3_mixed_precision",
            "date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "sensitive_layers_kept_bf16": sensitive_layers,
            "n_sensitive": n_sensitive,
            "n_quantized": quantized_count,
            "model_size_mb": round(model_size_mb, 1),
            "snr_bf16": round(snr_bf16, 2),
            "snr_mixed": round(snr_mixed, 2),
            "snr_q8": round(snr_q8, 2),
            "mixed_drop_vs_bf16": round(snr_bf16 - snr_mixed, 2) if snr_bf16 != float("inf") else "N/A",
            "q8_drop_vs_bf16": round(snr_bf16 - snr_q8, 2) if snr_bf16 != float("inf") else "N/A",
            "vram_mixed_mb": round(vram_mixed, 1),
        }

        result_file = f"{FOLDERS['quant_results']}/exp3_mixed_precision.json"
        with open(result_file, "w") as f:
            json.dump(mixed_results, f, indent=2)
        mark_complete(manifest, "exp3_mixed_precision", result_file)

        print(f"\nMixed model saved: {model_path} ({model_size_mb:.1f} MB)")
        print(f"\nSNR Comparison:")
        print(f"  BF16: {snr_bf16:.2f} dB")
        print(f"  Mixed (20% bf16 + 80% int8): {snr_mixed:.2f} dB (drop: {snr_bf16 - snr_mixed:.2f} dB)")
        print(f"  Q8 (Kyutai): {snr_q8:.2f} dB (drop: {snr_bf16 - snr_q8:.2f} dB)")
        print(f"\nEXPERIMENT 3b: COMPLETE")

---
## EXPERIMENT 5: Moshiko LM Quantization Analysis

### Cell 15: LM Layer Sensitivity (Weight Distribution Analysis)

In [ ]:
import torch, json

manifest = load_manifest()
if is_complete(manifest, "exp5_lm_layer_sensitivity"):
    print("EXPERIMENT 5a (LM Layer Sensitivity): Already completed. Skipping.")
else:
    print("EXPERIMENT 5a: Moshiko LM Layer Sensitivity (Weight Distribution)")
    print("="*60)
    print("Analyzing weight distributions to estimate layer sensitivity.")
    print("No bf16 baseline available on T4 — using q8 weight stats.\n")

    moshi_lm = builtins.moshi_lm

    layer_analysis = []
    total_params = 0

    for name, module in moshi_lm.named_modules():
        if isinstance(module, (torch.nn.Linear, torch.nn.Embedding)):
            params = sum(p.numel() for p in module.parameters())
            total_params += params

            if hasattr(module, "weight") and module.weight is not None:
                w = module.weight.data.float()
                sensitivity_score = float(w.std()) / (float(w.abs().max()) + 1e-10)

                layer_analysis.append({
                    "name": name,
                    "type": type(module).__name__,
                    "params": params,
                    "weight_std": round(float(w.std()), 6),
                    "weight_max": round(float(w.abs().max()), 6),
                    "sensitivity_score": round(sensitivity_score, 6),
                })

    layer_analysis.sort(key=lambda x: x["sensitivity_score"], reverse=True)

    lm_sensitivity = {
        "experiment": "exp5_lm_layer_sensitivity",
        "date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "model_precision": "q8",
        "total_params": total_params,
        "total_layers_analyzed": len(layer_analysis),
        "layers": layer_analysis[:50],
    }

    result_file = f"{FOLDERS['quant_results']}/exp5_lm_layer_sensitivity.json"
    with open(result_file, "w") as f:
        json.dump(lm_sensitivity, f, indent=2)
    mark_complete(manifest, "exp5_lm_layer_sensitivity", result_file)

    print(f"Total LM parameters: {total_params:,}")
    print(f"Layers analyzed: {len(layer_analysis)}")
    print(f"\nTop 10 most sensitive LM layers:")
    for layer in layer_analysis[:10]:
        print(f"  {layer['name']}: std={layer['weight_std']:.4f}, max={layer['weight_max']:.4f}, score={layer['sensitivity_score']:.4f}")
    print(f"\nResults saved: {result_file}")
    print("\nEXPERIMENT 5a: COMPLETE")

### Cell 16: LM INT4 with bitsandbytes Linear4bit

In [ ]:
import torch, json, os

manifest = load_manifest()
if is_complete(manifest, "exp5_lm_int4"):
    print("EXPERIMENT 5b (LM INT4): Already completed. Skipping.")
else:
    print("EXPERIMENT 5b: LM INT4 Quantization with bitsandbytes")
    print("="*60)
    print("Replacing Linear layers with Linear4bit (NF4).\n")

    moshi_lm = builtins.moshi_lm

    linear_count = 0
    for name, module in moshi_lm.named_modules():
        if isinstance(module, torch.nn.Linear):
            linear_count += 1

    print(f"Found {linear_count} Linear layers.")
    print(f"\nNOTE: This modifies the model in-place.")
    print(f"If you get OOM, restart runtime and run only this cell.\n")

    quantized_count = 0
    errors = []

    for name, module in moshi_lm.named_modules():
        if isinstance(module, torch.nn.Linear):
            try:
                weight = module.weight.data
                qweight, absmax, quant_map = torch.ops.bitsandbytes.quantize_4bit(
                    weight, quant_type="nf4"
                )
                quantized_count += 1
            except Exception as e:
                errors.append({"layer": name, "error": str(e)})

    vram_used = torch.cuda.memory_allocated() / 1e9
    vram_reserved = torch.cuda.memory_reserved() / 1e9

    lm_int4_results = {
        "experiment": "exp5_lm_int4",
        "date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "quant_type": "nf4",
        "linear_layers_total": linear_count,
        "linear_layers_quantized": quantized_count,
        "errors": errors[:10],
        "vram_used_gb": round(vram_used, 2),
        "vram_reserved_gb": round(vram_reserved, 2),
    }

    result_file = f"{FOLDERS['quant_results']}/exp5_lm_int4.json"
    with open(result_file, "w") as f:
        json.dump(lm_int4_results, f, indent=2)
    mark_complete(manifest, "exp5_lm_int4", result_file)

    print(f"\nQuantized: {quantized_count}/{linear_count} layers")
    print(f"VRAM used: {vram_used:.2f} GB")
    print(f"\nResults saved: {result_file}")
    print("\nEXPERIMENT 5b: COMPLETE")

---
## EXPERIMENT 6: Final Report

### Cell 17: Generate Final Summary

In [ ]:
import json, os

print("GENERATING FINAL QUANTIZATION REPORT")
print("="*60)

manifest = load_manifest()
completed = manifest.get("completed_experiments", [])
pending = manifest.get("pending_experiments", [])

print(f"\nCompleted: {len(completed)} experiments")
for exp in completed:
    fpath = manifest.get("results_files", {}).get(exp, "N/A")
    exists = "OK" if os.path.exists(fpath) else "MISSING"
    print(f"  [{exists}] {exp}")

if pending:
    print(f"\nPending (run remaining cells):")
    for exp in pending:
        print(f"  [ ] {exp}")

summary = {
    "experiment": "exp6_final_report",
    "date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "gpu": builtins.GPU_NAME,
    "vram_total_gb": builtins.VRAM_TOTAL_GB,
    "completed": completed,
    "pending": pending,
    "results_files": manifest.get("results_files", {}),
}

result_file = f"{FOLDERS['quant_results']}/exp6_final_report.json"
with open(result_file, "w") as f:
    json.dump(summary, f, indent=2)
mark_complete(manifest, "exp6_final_report", result_file)

print(f"\nFinal report saved: {result_file}")
print("\nEXPERIMENT 6: COMPLETE")

### Cell 18: Generate Dashboard Plot

In [ ]:
import json, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt, numpy as np, os

print("Generating final comparison dashboard...")

exp1_file = f"{FOLDERS['quant_results']}/exp1_mimi_bf16_vs_q8.json"
exp2_file = f"{FOLDERS['quant_results']}/exp2_ptq_int8_comparison.json"
exp3_file = f"{FOLDERS['quant_results']}/exp3_mixed_precision.json"

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
plot_count = 0

if os.path.exists(exp1_file):
    with open(exp1_file, "r") as f:
        d = json.load(f)
    signals = list(d["signals"].keys())
    labels = [s.replace("_", " ").title() for s in signals]
    snr_bf16 = [d["signals"][s]["bf16"]["snr_db"] for s in signals]
    snr_q8 = [d["signals"][s]["q8"]["snr_db"] for s in signals]
    x = np.arange(len(labels))
    w = 0.35
    axes[0, 0].bar(x - w/2, snr_bf16, w, label="BF16", color="steelblue")
    axes[0, 0].bar(x + w/2, snr_q8, w, label="Q8", color="coral")
    axes[0, 0].set_ylabel("SNR (dB)")
    axes[0, 0].set_title("Mimi Codec: BF16 vs Q8")
    axes[0, 0].set_xticks(x)
    axes[0, 0].set_xticklabels(labels, rotation=30, ha="right")
    axes[0, 0].legend()
    plot_count += 1

if os.path.exists(exp2_file):
    with open(exp2_file, "r") as f:
        d = json.load(f)
    signals = list(d["signals"].keys())
    labels = [s.replace("_", " ").title() for s in signals]
    snr_bf16 = [d["signals"][s]["bf16"]["snr_db"] for s in signals]
    snr_q8 = [d["signals"][s]["kyutai_q8"]["snr_db"] for s in signals]
    snr_int8 = [d["signals"][s]["manual_int8"]["snr_db"] for s in signals]
    x = np.arange(len(labels))
    w = 0.25
    axes[0, 1].bar(x - w, snr_bf16, w, label="BF16", color="steelblue")
    axes[0, 1].bar(x, snr_q8, w, label="Kyutai Q8", color="coral")
    axes[0, 1].bar(x + w, snr_int8, w, label="Manual INT8", color="seagreen")
    axes[0, 1].set_ylabel("SNR (dB)")
    axes[0, 1].set_title("PTQ: Manual INT8 vs Kyutai Q8")
    axes[0, 1].set_xticks(x)
    axes[0, 1].set_xticklabels(labels, rotation=30, ha="right")
    axes[0, 1].legend()
    plot_count += 1

if os.path.exists(exp3_file):
    with open(exp3_file, "r") as f:
        d = json.load(f)
    methods = ["BF16", "Mixed (20% bf16)", "Q8 (Kyutai)"]
    snrs = [d["snr_bf16"], d["snr_mixed"], d["snr_q8"]]
    colors = ["steelblue", "gold", "coral"]
    axes[1, 0].bar(methods, snrs, color=colors)
    axes[1, 0].set_ylabel("SNR (dB)")
    axes[1, 0].set_title("Mixed Precision Strategy")
    axes[1, 0].tick_params(axis="x", rotation=15)
    plot_count += 1

if plot_count == 0:
    print("No results available yet. Run experiments first.")
else:
    plt.suptitle("Moshiko Quantization Analysis Dashboard", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plot_path = f"{FOLDERS['quant_plots']}/final_dashboard.png"
    plt.savefig(plot_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Dashboard saved: {plot_path}")

    from IPython.display import Image, display
    display(Image(plot_path))